# DS4200 Final Project — Visualizations (Final)
All five visualizations built from `flights_cleaned.csv`. Run `preprocess.py` first.

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
import json, warnings
warnings.filterwarnings('ignore')
alt.data_transformers.disable_max_rows()

# ── Shared palette ────────────────────────────────────────────────────────────
PALETTE = {
    'Late Aircraft': '#e8913a',
    'Carrier':       '#5b8abf',
    'Weather':       '#6dbcb0',
    'NAS':           '#f2c45a',
    'Security':      '#c47ba0',
}
ACCENT       = '#e8913a'
CAUSE_ORDER  = ['Late Aircraft', 'Carrier', 'Weather', 'NAS', 'Security']
TOD_COLORS   = {
    'Morning':   '#5b8abf',
    'Afternoon': '#f2c45a',
    'Evening':   '#e8913a',
    'Night':     '#c47ba0',
}
TOD_ORDER = ['Morning', 'Afternoon', 'Evening', 'Night']

# ── Load cleaned data ─────────────────────────────────────────────────────────
df = pd.read_csv('flights_cleaned.csv', low_memory=False)
print(f"Loaded: {len(df):,} rows × {df.shape[1]} cols")
print(df.dtypes[['AIRLINE_NAME','ARRIVAL_DELAY','DEPARTURE_DELAY',
                  'hour','time_of_day','delayed','primary_delay_cause']].to_string())

## Viz 1 — Airline On-Time Performance (Altair, static)

In [ ]:
# ── Viz 1 data prep ───────────────────────────────────────────────────────────
CAUSE_COLS = {
    'LATE_AIRCRAFT_DELAY': 'Late Aircraft',
    'AIRLINE_DELAY':       'Carrier',
    'WEATHER_DELAY':       'Weather',
    'AIR_SYSTEM_DELAY':    'NAS',
    'SECURITY_DELAY':      'Security',
}

# % flights delayed >15 min per airline
total   = df.groupby('AIRLINE_NAME').size().rename('total')
delayed = df[df['ARRIVAL_DELAY'] > 15].groupby('AIRLINE_NAME').size().rename('delayed_ct')
pct_del = (delayed / total * 100).rename('pct_delayed')

# Sum each delay-cause column per airline (among ALL flights; NaN → 0 for summing)
cause_sums = (
    df.groupby('AIRLINE_NAME')[list(CAUSE_COLS.keys())]
    .sum()
    .rename(columns=CAUSE_COLS)
)
# Row-normalise → proportion of delay-cause minutes
cause_props = cause_sums.div(cause_sums.sum(axis=1), axis=0).fillna(0)

# Segment width = pct_delayed × proportion → "% of ALL flights from this cause"
seg = cause_props.multiply(pct_del, axis=0).reset_index()
long = seg.melt(id_vars='AIRLINE_NAME', var_name='Delay Cause', value_name='pct_of_flights')
long = long.merge(pct_del.reset_index(), on='AIRLINE_NAME')

# Sort order: best (lowest % delayed) at TOP, worst at BOTTOM
airline_order = (
    pct_del.sort_values(ascending=True).index.tolist()   # ascending → best first (top)
)

overall_avg = pct_del.mean()
print(f"Overall avg delay rate: {overall_avg:.1f}%")
print(f"Airline order (top→bottom): {airline_order}")
long.head()

In [ ]:
# ── Viz 1 chart ───────────────────────────────────────────────────────────────
color_scale = alt.Scale(
    domain=CAUSE_ORDER,
    range=[PALETTE[c] for c in CAUSE_ORDER]
)

bars = (
    alt.Chart(long)
    .mark_bar(height=22)
    .encode(
        x=alt.X('pct_of_flights:Q', stack='zero',
                title='% of All Flights Delayed >15 min (by Cause)',
                axis=alt.Axis(format='.1f', labelFontSize=11, titleFontSize=12)),
        y=alt.Y('AIRLINE_NAME:N', sort=airline_order,
                title=None,
                axis=alt.Axis(labelFontSize=11)),
        color=alt.Color('Delay Cause:N', scale=color_scale,
                        sort=CAUSE_ORDER,
                        legend=alt.Legend(title='Delay Cause', orient='bottom',
                                          direction='horizontal', columns=5,
                                          labelFontSize=11, titleFontSize=11)),
        order=alt.Order('Delay Cause:N', sort='ascending'),
        tooltip=[
            alt.Tooltip('AIRLINE_NAME:N',    title='Airline'),
            alt.Tooltip('Delay Cause:N',     title='Cause'),
            alt.Tooltip('pct_of_flights:Q',  title='% of All Flights', format='.2f'),
            alt.Tooltip('pct_delayed:Q',     title='Total % Delayed',  format='.1f'),
        ]
    )
)

rule = (
    alt.Chart(pd.DataFrame({'x': [overall_avg]}))
    .mark_rule(strokeDash=[5, 4], strokeWidth=1.5, color='#555')
    .encode(x='x:Q')
)
label = (
    alt.Chart(pd.DataFrame({'x': [overall_avg],
                             'label': [f"Avg: {overall_avg:.1f}%"]}))
    .mark_text(align='left', dx=4, dy=-8, fontSize=10, color='#555')
    .encode(x='x:Q', y=alt.value(8), text='label:N')
)

viz1 = (
    (bars + rule + label)
    .properties(
        title=alt.TitleParams(
            text='Airline On-Time Performance — Summer 2015',
            subtitle='Bar length = % of all flights delayed. Segments show breakdown by delay cause.',
            fontSize=14, subtitleFontSize=11, anchor='start'),
        width=620, height=320
    )
    .configure(background='white')
    .configure_view(strokeWidth=0)
    .configure_axis(grid=False)
)

viz1.save('viz1.html')
print("Saved → viz1.html")
viz1

## Viz 2 — Delay Heatmap: Hour × Day of Week (Altair, interactive)

In [ ]:
# ── Viz 2 data prep ───────────────────────────────────────────────────────────
from itertools import product as iproduct

DOW_MAP   = {1:'Mon', 2:'Tue', 3:'Wed', 4:'Thu', 5:'Fri', 6:'Sat', 7:'Sun'}
DOW_ORDER = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

df['DOW'] = df['DAY_OF_WEEK'].map(DOW_MAP)

heat = (
    df.groupby(['DOW', 'hour'])
    .agg(avg_delay=('DEPARTURE_DELAY','mean'), flight_cnt=('DEPARTURE_DELAY','count'))
    .reset_index()
)

# Fill every (DOW, hour) cell
full_grid = pd.DataFrame(list(iproduct(DOW_ORDER, range(24))), columns=['DOW','hour'])
heat = full_grid.merge(heat, on=['DOW','hour'], how='left').fillna(0)
heat['flight_cnt'] = heat['flight_cnt'].astype(int)
heat['avg_delay']  = heat['avg_delay'].round(1)
heat['hour_label'] = heat['hour'].apply(
    lambda h: ('12 AM' if h==0 else f'{h} AM' if h<12
               else '12 PM' if h==12 else f'{h-12} PM'))

print(f"Heatmap grid: {len(heat)} cells")
print(f"Delay range: {heat['avg_delay'].min()} – {heat['avg_delay'].max()} min")

In [ ]:
# ── Viz 2 chart ───────────────────────────────────────────────────────────────
viz2 = (
    alt.Chart(heat)
    .mark_rect(stroke='white', strokeWidth=0.4)
    .encode(
        x=alt.X('DOW:O', sort=DOW_ORDER, title='Day of Week',
                axis=alt.Axis(labelFontSize=11, titleFontSize=12, labelAngle=0)),
        y=alt.Y('hour:O', sort='descending', title='Hour of Day (Scheduled Departure)',
                axis=alt.Axis(labelFontSize=10, titleFontSize=12,
                              values=list(range(0,24,3)),
                              labelExpr=("datum.value==0?'12 AM':datum.value==12?'12 PM':"
                                         "datum.value<12?datum.value+' AM':(datum.value-12)+' PM'"))),
        color=alt.Color('avg_delay:Q', title='Avg Dep. Delay (min)',
                        scale=alt.Scale(scheme='oranges', domainMin=-3),
                        legend=alt.Legend(orient='right', gradientLength=180,
                                          labelFontSize=10, titleFontSize=11)),
        tooltip=[
            alt.Tooltip('DOW:O',        title='Day'),
            alt.Tooltip('hour_label:N', title='Hour'),
            alt.Tooltip('avg_delay:Q',  title='Avg Delay (min)', format='.1f'),
            alt.Tooltip('flight_cnt:Q', title='Flights',         format=','),
        ]
    )
    .properties(
        title=alt.TitleParams(
            text='Average Departure Delay by Hour and Day of Week',
            subtitle='Summer 2015 · Top-20 airports · Darker = longer delay',
            fontSize=14, subtitleFontSize=11, anchor='start'),
        width=460, height=440
    )
    .configure(background='white')
    .configure_view(strokeWidth=0)
)

viz2.save('viz2.html')
print("Saved → viz2.html")
viz2

## Viz 3 — Distance vs. Delay with Linked Histogram (Altair, interactive)

In [ ]:
# ── Viz 3 data prep ───────────────────────────────────────────────────────────
viz3_df = df[df['ARRIVAL_DELAY'].notna() & df['DISTANCE'].notna()].copy()
viz3_df = viz3_df[['DISTANCE','ARRIVAL_DELAY','time_of_day','AIRLINE_NAME']].reset_index(drop=True)

print(f"Viz 3 rows: {len(viz3_df):,}")
print(viz3_df['time_of_day'].value_counts().to_string())

In [ ]:
# ── Viz 3 chart (hconcat: scatter + histogram) ────────────────────────────────
brush = alt.selection_interval(encodings=['x','y'], name='brush')

tod_scale = alt.Scale(
    domain=TOD_ORDER,
    range=[TOD_COLORS[t] for t in TOD_ORDER]
)

scatter = (
    alt.Chart(viz3_df)
    .mark_circle(size=20, opacity=0.5)
    .encode(
        x=alt.X('DISTANCE:Q', title='Flight Distance (miles)',
                axis=alt.Axis(labelFontSize=10, titleFontSize=11)),
        y=alt.Y('ARRIVAL_DELAY:Q', title='Arrival Delay (min)',
                scale=alt.Scale(domain=[-60, 200]),
                axis=alt.Axis(labelFontSize=10, titleFontSize=11)),
        color=alt.condition(
            brush,
            alt.Color('time_of_day:N', scale=tod_scale,
                      sort=TOD_ORDER,
                      legend=alt.Legend(title='Time of Day', orient='bottom',
                                        direction='horizontal', columns=4,
                                        labelFontSize=10, titleFontSize=11)),
            alt.value('#cccccc')
        ),
        tooltip=[
            alt.Tooltip('AIRLINE_NAME:N',  title='Airline'),
            alt.Tooltip('DISTANCE:Q',      title='Distance (mi)', format=','),
            alt.Tooltip('ARRIVAL_DELAY:Q', title='Arrival Delay (min)', format='.0f'),
            alt.Tooltip('time_of_day:N',   title='Time of Day'),
        ]
    )
    .add_params(brush)
    .properties(
        title=alt.TitleParams(text='Distance vs. Arrival Delay',
                              subtitle='Drag to select — histogram updates',
                              fontSize=13, subtitleFontSize=10, anchor='start'),
        width=380, height=320
    )
)

zero_rule = (
    alt.Chart(pd.DataFrame({'y':[0]}))
    .mark_rule(strokeDash=[4,3], strokeWidth=1, color='#aaa')
    .encode(y='y:Q')
)

histogram = (
    alt.Chart(viz3_df)
    .mark_bar(color=ACCENT, opacity=0.85)
    .encode(
        x=alt.X('ARRIVAL_DELAY:Q', bin=alt.Bin(step=15),
                title='Arrival Delay (min)',
                axis=alt.Axis(labelFontSize=10, titleFontSize=11)),
        y=alt.Y('count():Q', title='Number of Flights',
                axis=alt.Axis(labelFontSize=10, titleFontSize=11)),
        tooltip=[
            alt.Tooltip('ARRIVAL_DELAY:Q', bin=alt.Bin(step=15), title='Delay bin (min)'),
            alt.Tooltip('count():Q', title='Flights', format=','),
        ]
    )
    .transform_filter(brush)
    .properties(
        title=alt.TitleParams(text='Delay Distribution (selected)',
                              subtitle='Updates from brush above',
                              fontSize=13, subtitleFontSize=10, anchor='start'),
        width=260, height=320
    )
)

viz3 = (
    alt.hconcat(scatter + zero_rule, histogram, spacing=24)
    .configure(background='white')
    .configure_view(strokeWidth=0)
    .configure_axis(grid=False)
)

viz3.save('viz3.html')
print("Saved → viz3.html")
viz3

## Viz 4 — U.S. Airport Delay Map (D3, interactive)

In [ ]:
# ── Viz 4 data prep → inline JSON ────────────────────────────────────────────
apt_stats = (
    df.groupby(['ORIGIN_AIRPORT','ORIGIN_AIRPORT_NAME','ORIGIN_CITY',
                'ORIGIN_STATE','ORIGIN_LAT','ORIGIN_LON'])
    .agg(total_flights  = ('FLIGHT_NUMBER','count'),
         avg_dep_delay  = ('DEPARTURE_DELAY','mean'))
    .reset_index()
)

top_carrier = (
    df.groupby(['ORIGIN_AIRPORT','AIRLINE_NAME'])
    .size().reset_index(name='cnt')
    .sort_values('cnt', ascending=False)
    .drop_duplicates('ORIGIN_AIRPORT')
    .rename(columns={'AIRLINE_NAME':'top_carrier'})[['ORIGIN_AIRPORT','top_carrier']]
)

apt_stats = apt_stats.merge(top_carrier, on='ORIGIN_AIRPORT', how='left')
apt_stats['avg_dep_delay'] = apt_stats['avg_dep_delay'].round(2)
apt_stats = apt_stats.dropna(subset=['ORIGIN_LAT','ORIGIN_LON'])

# Inline JSON for D3
apt_json = apt_stats.rename(columns={
    'ORIGIN_AIRPORT':      'iata',
    'ORIGIN_AIRPORT_NAME': 'name',
    'ORIGIN_CITY':         'city',
    'ORIGIN_STATE':        'state',
    'ORIGIN_LAT':          'lat',
    'ORIGIN_LON':          'lon',
    'total_flights':       'flights',
    'avg_dep_delay':       'avg_delay',
}).to_dict(orient='records')

print(f"Airports in map: {len(apt_json)}")
for a in sorted(apt_json, key=lambda x: -x['flights'])[:5]:
    print(f"  {a['iata']:4s} {a['city']:20s} flights={a['flights']:,}  avg_delay={a['avg_delay']:.1f} min")

In [ ]:
# ── Write viz4.html ───────────────────────────────────────────────────────────
APT_DATA = json.dumps(apt_json)

viz4_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<title>Viz 4 — U.S. Airport Delay Map</title>
<script src="https://d3js.org/d3.v7.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/topojson@3/dist/topojson.min.js"></script>
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{background:#ffffff;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;
        color:#333;display:flex;flex-direction:column;align-items:center;padding:20px 16px 36px}}
  h2{{font-size:16px;font-weight:600;color:#1a1a1a;width:100%;max-width:960px;margin-bottom:4px}}
  .sub{{font-size:12px;color:#666;width:100%;max-width:960px;margin-bottom:14px}}
  #map-wrap{{width:100%;max-width:960px;border-radius:8px;overflow:hidden;border:1px solid #ddd}}
  svg{{display:block;width:100%;background:#eef3f8}}
  .state{{fill:#dce8d4;stroke:#b0c4b0;stroke-width:0.7px}}
  .bubble{{stroke:#fff;stroke-width:0.6px;cursor:pointer}}
  .bubble:hover{{stroke:#222;stroke-width:1.8px}}
  #tip{{position:fixed;pointer-events:none;background:rgba(20,20,20,0.9);
        color:#f0f0f0;padding:10px 14px;border-radius:6px;font-size:12.5px;
        line-height:1.7;max-width:220px;display:none;z-index:999;
        border:1px solid #444}}
  #tip .code{{font-size:15px;font-weight:700;color:#e8913a;letter-spacing:1px}}
  .leg{{display:flex;align-items:flex-start;gap:36px;margin-top:12px;
        width:100%;max-width:960px}}
  .leg-title{{font-size:11px;font-weight:600;color:#555;margin-bottom:4px}}
  canvas{{border-radius:3px}}
  .cb-labels{{display:flex;justify-content:space-between;width:200px;
              font-size:10px;color:#777;margin-top:2px}}
  .sz-svg{{overflow:visible}}
  .hint{{font-size:11px;color:#999;margin-top:8px}}
</style>
</head>
<body>
<h2>U.S. Airport On-Time Performance — Summer 2015</h2>
<p class="sub">Bubble size = total departures &nbsp;·&nbsp; Color = avg departure delay &nbsp;·&nbsp; Scroll to zoom &nbsp;·&nbsp; Drag to pan &nbsp;·&nbsp; Hover for details</p>
<div id="map-wrap"><svg id="map" viewBox="0 0 960 560" preserveAspectRatio="xMidYMid meet"></svg></div>
<div class="leg">
  <div>
    <div class="leg-title">Avg Departure Delay (min)</div>
    <canvas id="cb" width="200" height="11"></canvas>
    <div class="cb-labels"><span id="cb-lo"></span><span>0</span><span id="cb-hi"></span></div>
  </div>
  <div>
    <div class="leg-title">Total Departures (sample)</div>
    <div id="sz-leg"></div>
  </div>
</div>
<p class="hint">Double-click to reset zoom</p>
<div id="tip"></div>
<script>
const AIRPORTS = {APT_DATA};
const W=960, H=560;
const svg = d3.select("#map");
const g   = svg.append("g");
const proj = d3.geoAlbersUsa().scale(1280).translate([W/2,H/2]);
const path = d3.geoPath().projection(proj);

const zoom = d3.zoom().scaleExtent([1,10]).on("zoom", ev=>{{
  g.attr("transform", ev.transform);
  g.selectAll(".bubble").attr("stroke-width", 0.6/ev.transform.k);
  g.selectAll(".state").attr("stroke-width", 0.7/ev.transform.k);
}});
svg.call(zoom);
svg.on("dblclick.zoom", ()=> svg.transition().duration(500).call(zoom.transform, d3.zoomIdentity));

const tip = d3.select("#tip");
function show(ev,d){{
  const s = d.avg_delay>=0 ? `+${{d.avg_delay.toFixed(1)}} min late` : `${{Math.abs(d.avg_delay).toFixed(1)}} min early`;
  tip.style("display","block").html(
    `<span class="code">${{d.iata}}</span><br/>${{d.name}}<br/>${{d.city}}, ${{d.state}}<br/>
     <strong>Flights:</strong> ${{d3.format(",")(d.flights)}}<br/>
     <strong>Avg Delay:</strong> ${{s}}<br/>
     <strong>Top Carrier:</strong> ${{d.top_carrier||"N/A"}}`
  );
  move(ev);
}}
function move(ev){{ tip.style("left",(ev.clientX+14)+"px").style("top",(ev.clientY-10)+"px"); }}
function hide(){{ tip.style("display","none"); }}

const valid = AIRPORTS.filter(d=>proj([d.lon,d.lat])!==null);
const maxF  = d3.max(valid, d=>d.flights);
const rScale = d3.scaleSqrt().domain([0,maxF]).range([3,28]);
const delays = d3.extent(valid, d=>d.avg_delay);
const cScale = d3.scaleSequential()
  .domain([delays[0], 35])
  .interpolator(d3.interpolateRgbBasis(["#5b8abf","#f5f0e8","#e8913a","#c0392b"]));

d3.json("https://cdn.jsdelivr.net/npm/us-atlas@3/states-10m.json").then(us=>{{
  g.append("g").selectAll("path")
    .data(topojson.feature(us,us.objects.states).features)
    .join("path").attr("class","state").attr("d",path);
  g.append("path")
    .datum(topojson.mesh(us,us.objects.states,(a,b)=>a!==b))
    .attr("fill","none").attr("stroke","#b0c4b0").attr("stroke-width","0.7").attr("d",path);

  g.append("g").selectAll("circle")
    .data([...valid].sort((a,b)=>b.flights-a.flights))
    .join("circle")
    .attr("class","bubble")
    .attr("cx", d=>proj([d.lon,d.lat])[0])
    .attr("cy", d=>proj([d.lon,d.lat])[1])
    .attr("r",  d=>rScale(d.flights))
    .attr("fill", d=>cScale(d.avg_delay))
    .attr("opacity",0.85)
    .on("mouseover",show).on("mousemove",move).on("mouseout",hide);

  // Color bar
  const cv=document.getElementById("cb"), cx=cv.getContext("2d");
  const gr=cx.createLinearGradient(0,0,200,0);
  for(let i=0;i<=10;i++){{ const t=i/10; gr.addColorStop(t,cScale(delays[0]+t*(35-delays[0]))); }}
  cx.fillStyle=gr; cx.fillRect(0,0,200,11);
  document.getElementById("cb-lo").textContent=delays[0].toFixed(1)+" min";
  document.getElementById("cb-hi").textContent="35+ min";

  // Size legend
  const ticks=[100,300,600];
  const szSvg=d3.select("#sz-leg").append("svg").attr("class","sz-svg").attr("width",200).attr("height",52);
  let ox=16;
  ticks.forEach(v=>{{
    const r=rScale(v*(maxF/1000));
    szSvg.append("circle").attr("cx",ox+r).attr("cy",28).attr("r",r)
      .attr("fill","#bbb").attr("stroke","#fff").attr("stroke-width",0.6);
    szSvg.append("text").attr("x",ox+r).attr("y",28+r+11)
      .attr("text-anchor","middle").attr("font-size","10px").attr("fill","#777")
      .text(v+"‰ of max");
    ox+=r*2+14;
  }});
}});
</script>
</body></html>"""

with open("viz4.html","w") as f:
    f.write(viz4_html)
print("Saved → viz4.html")

## Viz 5 — Daily Delay Trend by Airline (D3, interactive)

In [ ]:
# ── Viz 5 data prep → inline JSON ────────────────────────────────────────────
df['DATE'] = pd.to_datetime(
    df[['YEAR','MONTH','DAY']].rename(columns={'YEAR':'year','MONTH':'month','DAY':'day'})
).dt.strftime('%Y-%m-%d')

daily = (
    df.groupby(['DATE','AIRLINE_NAME'])
    .agg(avg_delay=('ARRIVAL_DELAY','mean'), flights=('ARRIVAL_DELAY','count'))
    .reset_index()
)
daily['avg_delay'] = daily['avg_delay'].round(2)

# Top-5 worst airlines over the summer
worst5 = (
    daily.groupby('AIRLINE_NAME')['avg_delay']
    .mean().sort_values(ascending=False).head(5).index.tolist()
)
all_airlines = (
    daily.groupby('AIRLINE_NAME')['avg_delay']
    .mean().sort_values(ascending=False).index.tolist()
)

v5_payload = {
    'airlines': all_airlines,
    'default':  worst5,
    'data': daily[['DATE','AIRLINE_NAME','avg_delay']].to_dict(orient='records')
}
V5_DATA = json.dumps(v5_payload)

print(f"Daily rows: {len(daily):,}")
print("Top-5 worst airlines (summer avg):")
for a in worst5:
    print(f"  {a}  ({daily[daily.AIRLINE_NAME==a].avg_delay.mean():.1f} min)")

In [ ]:
# ── Write viz5.html ───────────────────────────────────────────────────────────
viz5_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<title>Viz 5 — Daily Delay Trend</title>
<script src="https://d3js.org/d3.v7.min.js"></script>
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{background:#ffffff;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;
        color:#333;display:flex;flex-direction:column;align-items:center;padding:20px 16px 40px}}
  h2{{font-size:16px;font-weight:600;color:#1a1a1a;width:100%;max-width:920px;margin-bottom:4px}}
  .sub{{font-size:12px;color:#666;width:100%;max-width:920px;margin-bottom:12px}}
  /* dropdown */
  .dd-wrap{{position:relative;width:100%;max-width:920px;margin-bottom:12px}}
  #dd-btn{{background:#f5f5f5;border:1px solid #ccc;border-radius:5px;
           padding:7px 12px;font-size:13px;color:#333;cursor:pointer;
           display:flex;align-items:center;gap:8px;width:250px}}
  #dd-btn:hover{{border-color:#888}}
  #dd-btn .arr{{margin-left:auto;font-size:9px;color:#999}}
  #dd-panel{{display:none;position:absolute;top:calc(100% + 4px);left:0;
             background:#fff;border:1px solid #ccc;border-radius:6px;
             padding:6px 0;width:280px;max-height:320px;overflow-y:auto;
             z-index:100;box-shadow:0 4px 16px rgba(0,0,0,.12)}}
  #dd-panel.open{{display:block}}
  .dd-item{{display:flex;align-items:center;gap:8px;padding:5px 12px;
            cursor:pointer;font-size:12.5px;user-select:none}}
  .dd-item:hover{{background:#f5f5f5}}
  .dd-item input{{accent-color:#e8913a;width:13px;height:13px}}
  .sw{{width:9px;height:9px;border-radius:50%;flex-shrink:0}}
  .dd-actions{{display:flex;gap:8px;padding:5px 12px 2px}}
  .dd-actions button{{font-size:11px;color:#e8913a;background:none;border:none;
                      cursor:pointer;padding:0;text-decoration:underline}}
  hr.divider{{border:none;border-top:1px solid #eee;margin:4px 0}}
  /* chart */
  #chart-wrap{{width:100%;max-width:920px;background:#ffffff;
               border:1px solid #e0e0e0;border-radius:8px;padding:8px 0 4px}}
  .axis path,.axis line{{stroke:#ddd}}
  .axis text{{fill:#666;font-size:11px}}
  .grid line{{stroke:#f0f0f0;stroke-dasharray:3,3}}
  .grid path{{stroke-width:0}}
  .zero-line{{stroke:#ccc;stroke-dasharray:4,3;stroke-width:1px}}
  .hol-line{{stroke:#e8913a;stroke-dasharray:5,4;stroke-width:1.2px;opacity:.6}}
  .hol-lbl{{font-size:10px;fill:#e8913a;opacity:.7}}
  .airline-line{{fill:none;stroke-width:2px}}
  .hover-line{{stroke:#bbb;stroke-width:1px;pointer-events:none}}
  .hover-dot{{pointer-events:none}}
  /* tooltip */
  #tip{{position:fixed;pointer-events:none;background:rgba(20,20,20,0.88);
        color:#f0f0f0;padding:9px 13px;border-radius:6px;font-size:12px;
        line-height:1.75;max-width:240px;display:none;z-index:999}}
  .tt-date{{font-weight:700;color:#e8913a;margin-bottom:3px;font-size:13px}}
  .tt-row{{display:flex;justify-content:space-between;gap:14px}}
  .tt-sw{{display:inline-block;width:7px;height:7px;border-radius:50%;margin-right:5px}}
  /* legend */
  #legend{{display:flex;flex-wrap:wrap;gap:8px 18px;margin-top:10px;
           width:100%;max-width:920px;font-size:12px;color:#555}}
  .leg-item{{display:flex;align-items:center;gap:6px}}
  .leg-line{{width:20px;height:2.5px;border-radius:2px}}
</style>
</head>
<body>
<h2>Average Daily Arrival Delay by Airline — Summer 2015 (Jun–Aug)</h2>
<p class="sub">Select airlines to compare &nbsp;·&nbsp; Hover to see exact values &nbsp;·&nbsp; Orange dashed line = July 4th</p>

<div class="dd-wrap">
  <button id="dd-btn"><span id="dd-lbl">Airlines shown: 5</span><span class="arr">▼</span></button>
  <div id="dd-panel">
    <div class="dd-actions">
      <button id="sel-all">All</button>
      <button id="sel-none">Clear</button>
      <button id="sel-def">Reset</button>
    </div>
    <hr class="divider"/>
  </div>
</div>

<div id="chart-wrap"><svg id="chart"></svg></div>
<div id="legend"></div>
<div id="tip"></div>

<script>
const PAYLOAD = {V5_DATA};
const COLORS = ["#e8913a","#5b8abf","#6dbcb0","#f2c45a","#c47ba0",
                 "#59a14f","#edc948","#ff9da7","#9c755f","#bab0ac",
                 "#d37295","#499894","#86bcb6","#e15759"];

const ML={{top:28,right:24,bottom:50,left:58}};
const TW=880, TH=400;
const W=TW-ML.left-ML.right, H=TH-ML.top-ML.bottom;

const svg=d3.select("#chart").attr("width",TW).attr("height",TH);
const g=svg.append("g").attr("transform",`translate(${{ML.left}},${{ML.top}})`);
const tip=d3.select("#tip");

const allAirlines=PAYLOAD.airlines;
const defSet=new Set(PAYLOAD.default);
let selected=new Set(PAYLOAD.default);

const parseDate=d3.timeParse("%Y-%m-%d");
const raw=PAYLOAD.data.map(d=>({{date:parseDate(d.DATE),airline:d.AIRLINE_NAME,delay:d.avg_delay}}));
const byAirline=d3.group(raw, d=>d.airline);
const colorMap={{}};
allAirlines.forEach((a,i)=>colorMap[a]=COLORS[i%COLORS.length]);

const xScale=d3.scaleTime().domain([new Date("2015-06-01"),new Date("2015-08-31")]).range([0,W]);
const allDelays=raw.map(d=>d.delay);
const yScale=d3.scaleLinear().domain([d3.min(allDelays)-2, d3.max(allDelays)+4]).nice().range([H,0]);

g.append("g").attr("class","axis x-axis").attr("transform",`translate(0,${{H}})`)
  .call(d3.axisBottom(xScale).ticks(d3.timeWeek.every(2)).tickFormat(d3.timeFormat("%b %d")));
g.append("g").attr("class","axis y-axis")
  .call(d3.axisLeft(yScale).ticks(7).tickFormat(d=>d+" min"));

g.append("text").attr("x",W/2).attr("y",H+42).attr("text-anchor","middle")
  .attr("font-size","11px").attr("fill","#888").text("Date");
g.append("text").attr("transform","rotate(-90)").attr("x",-H/2).attr("y",-46)
  .attr("text-anchor","middle").attr("font-size","11px").attr("fill","#888")
  .text("Avg Arrival Delay (min)");

g.append("g").attr("class","grid").call(d3.axisLeft(yScale).ticks(7).tickSize(-W).tickFormat(""));
g.append("line").attr("class","zero-line").attr("x1",0).attr("x2",W)
  .attr("y1",yScale(0)).attr("y2",yScale(0));

const hx=xScale(new Date("2015-07-04"));
g.append("line").attr("class","hol-line").attr("x1",hx).attr("x2",hx).attr("y1",0).attr("y2",H);
g.append("text").attr("class","hol-lbl").attr("x",hx+3).attr("y",12).text("Jul 4");

const lineGen=d3.line().defined(d=>d.delay!=null&&!isNaN(d.delay))
  .x(d=>xScale(d.date)).y(d=>yScale(d.delay)).curve(d3.curveMonotoneX);
const linesG=g.append("g");

const hLine=g.append("line").attr("class","hover-line").attr("y1",0).attr("y2",H).style("display","none");
const dotsG=g.append("g");
const bisect=d3.bisector(d=>d.date).left;

g.append("rect").attr("width",W).attr("height",H).attr("fill","none")
  .attr("pointer-events","all")
  .on("mousemove", ev=>{{
    const [mx]=d3.pointer(ev);
    const hd=xScale.invert(mx);
    hLine.style("display",null).attr("x1",mx).attr("x2",mx);
    const rows=[];
    allAirlines.filter(a=>selected.has(a)).forEach(a=>{{
      const pts=(byAirline.get(a)||[]).sort((a,b)=>a.date-b.date);
      if(!pts.length) return;
      const i=bisect(pts,hd,1);
      const d0=pts[i-1],d1=pts[i];
      const d=!d1?d0:!d0?d1:hd-d0.date<d1.date-hd?d0:d1;
      if(d) rows.push({{airline:a,d,color:colorMap[a]}});
    }});
    dotsG.selectAll(".hover-dot").remove();
    rows.forEach(r=>{{
      if(r.d.delay==null||isNaN(r.d.delay)) return;
      dotsG.append("circle").attr("class","hover-dot")
        .attr("cx",xScale(r.d.date)).attr("cy",yScale(r.d.delay))
        .attr("r",4).attr("fill",r.color).attr("stroke","#fff").attr("stroke-width",1.5);
    }});
    if(!rows.length){{ tip.style("display","none"); return; }}
    const ds=d3.timeFormat("%a, %b %e")(rows[0].d.date);
    const rh=rows.sort((a,b)=>b.d.delay-a.d.delay)
      .map(r=>`<div class="tt-row"><span><span class="tt-sw" style="background:${{r.color}}"></span>${{r.airline.replace(/ Inc\.| Co\./g,"")}}</span><span><strong>${{r.d.delay>=0?"+":""}}${{r.d.delay.toFixed(1)}} min</strong></span></div>`)
      .join("");
    tip.style("display","block").html(`<div class="tt-date">${{ds}}</div>${{rh}}`);
    tip.style("left",(ev.clientX+14)+"px").style("top",(ev.clientY-10)+"px");
  }})
  .on("mouseleave",()=>{{
    hLine.style("display","none");
    dotsG.selectAll(".hover-dot").remove();
    tip.style("display","none");
  }});

function draw(){{
  const ld=allAirlines.filter(a=>selected.has(a));
  const paths=linesG.selectAll(".airline-line").data(ld,d=>d);
  paths.enter().append("path").attr("class","airline-line")
    .merge(paths).attr("stroke",d=>colorMap[d])
    .attr("d",d=>{{
      const pts=(byAirline.get(d)||[]).sort((a,b)=>a.date-b.date);
      return lineGen(pts);
    }});
  paths.exit().remove();
  const leg=d3.select("#legend");
  leg.selectAll(".leg-item").remove();
  ld.forEach(a=>{{
    const item=leg.append("div").attr("class","leg-item");
    item.append("div").attr("class","leg-line").style("background",colorMap[a]);
    item.append("span").text(a);
  }});
  document.getElementById("dd-lbl").textContent=`Airlines shown: ${{selected.size}}`;
}}

const panel=document.getElementById("dd-panel");
const btn=document.getElementById("dd-btn");
btn.addEventListener("click",e=>{{e.stopPropagation();panel.classList.toggle("open")}});
document.addEventListener("click",()=>panel.classList.remove("open"));
panel.addEventListener("click",e=>e.stopPropagation());

allAirlines.forEach(a=>{{
  const lbl=document.createElement("label"); lbl.className="dd-item";
  const cb=document.createElement("input"); cb.type="checkbox"; cb.checked=defSet.has(a);
  cb.addEventListener("change",()=>{{cb.checked?selected.add(a):selected.delete(a);draw()}});
  const sw=document.createElement("div"); sw.className="sw"; sw.style.background=colorMap[a];
  lbl.appendChild(cb); lbl.appendChild(sw); lbl.appendChild(document.createTextNode(a));
  panel.appendChild(lbl);
}});

document.getElementById("sel-all").onclick=()=>{{
  selected=new Set(allAirlines);
  panel.querySelectorAll("input").forEach(c=>c.checked=true); draw();
}};
document.getElementById("sel-none").onclick=()=>{{
  selected=new Set();
  panel.querySelectorAll("input").forEach(c=>c.checked=false); draw();
}};
document.getElementById("sel-def").onclick=()=>{{
  selected=new Set(PAYLOAD.default);
  panel.querySelectorAll(".dd-item").forEach(lbl=>{{
    lbl.querySelector("input").checked=defSet.has(lbl.textContent.trim());
  }}); draw();
}};

draw();
</script>
</body></html>"""

with open("viz5.html","w") as f:
    f.write(viz5_html)
print("Saved → viz5.html")